In [4]:
!pip install alpha_vantage pandas numpy requests yfinance
from alpha_vantage.timeseries import TimeSeries
import pandas as pd
import numpy as np

ALPHA_KEY = "0QHGOSAUAN8CMKJJ"  # Replace with your actual key

# Cache to avoid multiple API calls
prices_cache = {}

def get_prices_alpha(ticker):
    if ticker in prices_cache:
        return prices_cache[ticker]
    try:
        ts = TimeSeries(key=ALPHA_KEY, output_format='pandas')
        data, meta = ts.get_daily_adjusted(symbol=ticker, outputsize='full')
        prices = data['5. adjusted close']
        prices_cache[ticker] = prices
        return prices
    except Exception as e:
        print(f"Alpha Vantage failed for {ticker}: {e}")
        return pd.Series()  # return empty series to avoid crash

def get_var(ticker, confidence=0.95):
    prices = get_prices_alpha(ticker)
    
    if prices.empty:
        print(f"Warning: No price data for {ticker}. Setting VaR = 0")
        return 0

    returns = prices.pct_change().dropna()
    return np.percentile(returns, (1 - confidence) * 100)


Enter stock ticker (e.g., AAPL, MSFT, BHP.AX): AAPL


Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['AAPL']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')


ValueError: attempt to get argmax of an empty sequence

In [9]:
ticker = "AAPL"
print("VaR:", get_var(ticker))

Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['AAPL']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')


ValueError: attempt to get argmax of an empty sequence

In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class CreditWorthinessModel:
    """
    Comprehensive credit worthiness analysis model using Yahoo Finance data
    """
    
    def __init__(self, ticker):
        self.ticker = ticker.upper()
        self.stock = yf.Ticker(self.ticker)
        self.financials = None
        self.balance_sheet = None
        self.cashflow = None
        self.info = None
        self.historical_prices = None
        self.results = {}
        
    def fetch_data(self):
        """Fetch all necessary financial data"""
        print(f"Fetching data for {self.ticker}...")
        
        try:
            # Get financial statements
            self.financials = self.stock.financials
            self.balance_sheet = self.stock.balance_sheet
            self.cashflow = self.stock.cashflow
            self.info = self.stock.info
            
            # Get 5 years of historical price data
            end_date = datetime.now()
            start_date = end_date - timedelta(days=5*365)
            self.historical_prices = self.stock.history(start=start_date, end=end_date)
            
            print("Data fetched successfully!")
            return True
        except Exception as e:
            print(f"Error fetching data: {e}")
            return False
    
    def calculate_altman_z_score(self):
        """
        Calculate Altman Z-Score for credit risk assessment
        Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
        where:
        X1 = Working Capital / Total Assets
        X2 = Retained Earnings / Total Assets
        X3 = EBIT / Total Assets
        X4 = Market Value of Equity / Total Liabilities
        X5 = Sales / Total Assets
        """
        try:
            bs = self.balance_sheet.iloc[:, 0]  # Most recent balance sheet
            income = self.financials.iloc[:, 0]  # Most recent income statement
            
            # Extract values (handling missing data)
            total_assets = bs.get('Total Assets', 0)
            current_assets = bs.get('Current Assets', 0)
            current_liabilities = bs.get('Current Liabilities', 0)
            retained_earnings = bs.get('Retained Earnings', 0)
            ebit = income.get('EBIT', income.get('Operating Income', 0))
            total_liabilities = bs.get('Total Liabilities Net Minority Interest', 0)
            revenue = income.get('Total Revenue', 0)
            
            # Market cap from info
            market_cap = self.info.get('marketCap', 0)
            
            if total_assets == 0:
                return None, "Insufficient data for Altman Z-Score"
            
            # Calculate components
            X1 = (current_assets - current_liabilities) / total_assets
            X2 = retained_earnings / total_assets
            X3 = ebit / total_assets
            X4 = market_cap / total_liabilities if total_liabilities > 0 else 0
            X5 = revenue / total_assets
            
            # Calculate Z-Score
            z_score = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
            
            # Interpretation
            if z_score > 2.99:
                interpretation = "Safe Zone (Low bankruptcy risk)"
            elif z_score > 1.81:
                interpretation = "Grey Zone (Moderate risk)"
            else:
                interpretation = "Distress Zone (High bankruptcy risk)"
            
            self.results['altman_z_score'] = {
                'score': z_score,
                'interpretation': interpretation,
                'components': {
                    'X1_working_capital_ratio': X1,
                    'X2_retained_earnings_ratio': X2,
                    'X3_ebit_ratio': X3,
                    'X4_market_equity_ratio': X4,
                    'X5_asset_turnover': X5
                }
            }
            
            return z_score, interpretation
            
        except Exception as e:
            return None, f"Error calculating Altman Z-Score: {e}"
    
    def calculate_var_cvar(self, confidence_level=0.95):
        """
        Calculate Value at Risk (VaR) and Conditional VaR (CVaR)
        Using historical simulation method on stock returns
        """
        try:
            if self.historical_prices is None or len(self.historical_prices) < 2:
                return None, None, "Insufficient price data"
            
            # Calculate daily returns
            returns = self.historical_prices['Close'].pct_change().dropna()
            
            # Calculate VaR
            var = np.percentile(returns, (1 - confidence_level) * 100)
            
            # Calculate CVaR (Expected Shortfall)
            cvar = returns[returns <= var].mean()
            
            # Annualized metrics
            var_annual = var * np.sqrt(252)
            cvar_annual = cvar * np.sqrt(252)
            
            self.results['var_cvar'] = {
                'var_daily': var,
                'cvar_daily': cvar,
                'var_annual': var_annual,
                'cvar_annual': cvar_annual,
                'confidence_level': confidence_level,
                'interpretation': f"There is a {confidence_level*100}% chance that daily losses will not exceed {abs(var)*100:.2f}%"
            }
            
            return var, cvar, "Success"
            
        except Exception as e:
            return None, None, f"Error calculating VaR/CVaR: {e}"
    
    def calculate_liquidity_ratios(self):
        """Calculate various liquidity ratios"""
        try:
            bs = self.balance_sheet.iloc[:, 0]
            
            current_assets = bs.get('Current Assets', 0)
            current_liabilities = bs.get('Current Liabilities', 0)
            cash = bs.get('Cash And Cash Equivalents', 0)
            inventory = bs.get('Inventory', 0)
            
            current_ratio = current_assets / current_liabilities if current_liabilities > 0 else 0
            quick_ratio = (current_assets - inventory) / current_liabilities if current_liabilities > 0 else 0
            cash_ratio = cash / current_liabilities if current_liabilities > 0 else 0
            
            self.results['liquidity_ratios'] = {
                'current_ratio': current_ratio,
                'quick_ratio': quick_ratio,
                'cash_ratio': cash_ratio,
                'interpretation': {
                    'current_ratio': 'Good' if current_ratio > 1.5 else 'Moderate' if current_ratio > 1 else 'Poor',
                    'quick_ratio': 'Good' if quick_ratio > 1 else 'Moderate' if quick_ratio > 0.5 else 'Poor'
                }
            }
            
            return self.results['liquidity_ratios']
            
        except Exception as e:
            return f"Error calculating liquidity ratios: {e}"
    
    def calculate_leverage_ratios(self):
        """Calculate leverage and solvency ratios"""
        try:
            bs = self.balance_sheet.iloc[:, 0]
            income = self.financials.iloc[:, 0]
            
            total_debt = bs.get('Total Debt', 0)
            total_equity = bs.get('Total Equity Gross Minority Interest', 0)
            total_assets = bs.get('Total Assets', 0)
            ebit = income.get('EBIT', income.get('Operating Income', 0))
            interest_expense = income.get('Interest Expense', 0)
            
            debt_to_equity = total_debt / total_equity if total_equity > 0 else 0
            debt_to_assets = total_debt / total_assets if total_assets > 0 else 0
            equity_multiplier = total_assets / total_equity if total_equity > 0 else 0
            interest_coverage = ebit / abs(interest_expense) if interest_expense != 0 else 0
            
            self.results['leverage_ratios'] = {
                'debt_to_equity': debt_to_equity,
                'debt_to_assets': debt_to_assets,
                'equity_multiplier': equity_multiplier,
                'interest_coverage': interest_coverage,
                'interpretation': {
                    'debt_to_equity': 'Low leverage' if debt_to_equity < 1 else 'Moderate' if debt_to_equity < 2 else 'High leverage',
                    'interest_coverage': 'Strong' if interest_coverage > 5 else 'Adequate' if interest_coverage > 2.5 else 'Weak'
                }
            }
            
            return self.results['leverage_ratios']
            
        except Exception as e:
            return f"Error calculating leverage ratios: {e}"
    
    def calculate_profitability_ratios(self):
        """Calculate profitability metrics"""
        try:
            income = self.financials.iloc[:, 0]
            bs = self.balance_sheet.iloc[:, 0]
            
            revenue = income.get('Total Revenue', 0)
            gross_profit = income.get('Gross Profit', 0)
            operating_income = income.get('Operating Income', 0)
            net_income = income.get('Net Income', 0)
            total_assets = bs.get('Total Assets', 0)
            total_equity = bs.get('Total Equity Gross Minority Interest', 0)
            
            gross_margin = gross_profit / revenue if revenue > 0 else 0
            operating_margin = operating_income / revenue if revenue > 0 else 0
            net_margin = net_income / revenue if revenue > 0 else 0
            roa = net_income / total_assets if total_assets > 0 else 0
            roe = net_income / total_equity if total_equity > 0 else 0
            
            self.results['profitability_ratios'] = {
                'gross_margin': gross_margin,
                'operating_margin': operating_margin,
                'net_margin': net_margin,
                'roa': roa,
                'roe': roe,
                'interpretation': {
                    'net_margin': 'Strong' if net_margin > 0.15 else 'Moderate' if net_margin > 0.05 else 'Weak',
                    'roe': 'Excellent' if roe > 0.15 else 'Good' if roe > 0.10 else 'Below average'
                }
            }
            
            return self.results['profitability_ratios']
            
        except Exception as e:
            return f"Error calculating profitability ratios: {e}"
    
    def calculate_cashflow_metrics(self):
        """Calculate cash flow related metrics"""
        try:
            cf = self.cashflow.iloc[:, 0]
            income = self.financials.iloc[:, 0]
            bs = self.balance_sheet.iloc[:, 0]
            
            operating_cf = cf.get('Operating Cash Flow', 0)
            free_cf = cf.get('Free Cash Flow', 0)
            net_income = income.get('Net Income', 0)
            total_debt = bs.get('Total Debt', 0)
            
            ocf_to_net_income = operating_cf / net_income if net_income > 0 else 0
            fcf_to_debt = free_cf / total_debt if total_debt > 0 else 0
            
            self.results['cashflow_metrics'] = {
                'operating_cashflow': operating_cf,
                'free_cashflow': free_cf,
                'ocf_to_net_income': ocf_to_net_income,
                'fcf_to_debt': fcf_to_debt,
                'interpretation': {
                    'ocf_quality': 'Strong' if ocf_to_net_income > 1 else 'Moderate' if ocf_to_net_income > 0.8 else 'Weak',
                    'debt_coverage': 'Strong' if fcf_to_debt > 0.2 else 'Moderate' if fcf_to_debt > 0.1 else 'Weak'
                }
            }
            
            return self.results['cashflow_metrics']
            
        except Exception as e:
            return f"Error calculating cashflow metrics: {e}"
    
    def calculate_volatility_metrics(self):
        """Calculate stock volatility and beta"""
        try:
            returns = self.historical_prices['Close'].pct_change().dropna()
            
            # Historical volatility
            daily_vol = returns.std()
            annual_vol = daily_vol * np.sqrt(252)
            
            # Beta from Yahoo Finance
            beta = self.info.get('beta', None)
            
            # Sharpe Ratio (assuming risk-free rate of 4%)
            risk_free_rate = 0.04
            avg_return = returns.mean() * 252
            sharpe = (avg_return - risk_free_rate) / annual_vol if annual_vol > 0 else 0
            
            self.results['volatility_metrics'] = {
                'daily_volatility': daily_vol,
                'annual_volatility': annual_vol,
                'beta': beta,
                'sharpe_ratio': sharpe,
                'interpretation': {
                    'volatility': 'Low' if annual_vol < 0.2 else 'Moderate' if annual_vol < 0.4 else 'High',
                    'beta': 'Defensive' if beta and beta < 1 else 'Market' if beta and beta < 1.2 else 'Aggressive'
                }
            }
            
            return self.results['volatility_metrics']
            
        except Exception as e:
            return f"Error calculating volatility metrics: {e}"
    
    def calculate_credit_score(self):
        """
        Calculate overall credit score (0-100) based on all metrics
        Higher score = better creditworthiness
        """
        try:
            score = 50  # Base score
            
            # Altman Z-Score component (0-20 points)
            if 'altman_z_score' in self.results:
                z = self.results['altman_z_score']['score']
                if z > 2.99:
                    score += 20
                elif z > 1.81:
                    score += 10
                else:
                    score += 0
            
            # Liquidity component (0-15 points)
            if 'liquidity_ratios' in self.results:
                current_ratio = self.results['liquidity_ratios']['current_ratio']
                if current_ratio > 2:
                    score += 15
                elif current_ratio > 1:
                    score += 10
                else:
                    score += 5
            
            # Leverage component (0-20 points)
            if 'leverage_ratios' in self.results:
                d_to_e = self.results['leverage_ratios']['debt_to_equity']
                int_cov = self.results['leverage_ratios']['interest_coverage']
                
                if d_to_e < 1:
                    score += 10
                elif d_to_e < 2:
                    score += 5
                
                if int_cov > 5:
                    score += 10
                elif int_cov > 2.5:
                    score += 5
            
            # Profitability component (0-15 points)
            if 'profitability_ratios' in self.results:
                roe = self.results['profitability_ratios']['roe']
                net_margin = self.results['profitability_ratios']['net_margin']
                
                if roe > 0.15:
                    score += 8
                elif roe > 0.10:
                    score += 5
                
                if net_margin > 0.15:
                    score += 7
                elif net_margin > 0.05:
                    score += 4
            
            # Risk component (0-15 points) - lower risk is better
            if 'var_cvar' in self.results:
                var_annual = abs(self.results['var_cvar']['var_annual'])
                if var_annual < 0.15:
                    score += 15
                elif var_annual < 0.25:
                    score += 10
                else:
                    score += 5
            
            # Cash flow component (0-15 points)
            if 'cashflow_metrics' in self.results:
                ocf_ratio = self.results['cashflow_metrics']['ocf_to_net_income']
                if ocf_ratio > 1:
                    score += 15
                elif ocf_ratio > 0.8:
                    score += 10
                else:
                    score += 5
            
            # Cap at 100
            score = min(score, 100)
            
            # Rating
            if score >= 85:
                rating = "AAA - Excellent"
            elif score >= 75:
                rating = "AA - Very Good"
            elif score >= 65:
                rating = "A - Good"
            elif score >= 55:
                rating = "BBB - Adequate"
            elif score >= 45:
                rating = "BB - Moderate Risk"
            elif score >= 35:
                rating = "B - High Risk"
            else:
                rating = "CCC - Very High Risk"
            
            self.results['credit_score'] = {
                'score': score,
                'rating': rating
            }
            
            return score, rating
            
        except Exception as e:
            return None, f"Error calculating credit score: {e}"
    
    def run_full_analysis(self):
        """Run complete credit analysis"""
        if not self.fetch_data():
            return None
        
        print("\n" + "="*60)
        print(f"CREDIT WORTHINESS ANALYSIS: {self.ticker}")
        print("="*60)
        
        # Run all calculations
        print("\n1. Calculating Altman Z-Score...")
        self.calculate_altman_z_score()
        
        print("2. Calculating VaR and CVaR...")
        self.calculate_var_cvar()
        
        print("3. Calculating Liquidity Ratios...")
        self.calculate_liquidity_ratios()
        
        print("4. Calculating Leverage Ratios...")
        self.calculate_leverage_ratios()
        
        print("5. Calculating Profitability Ratios...")
        self.calculate_profitability_ratios()
        
        print("6. Calculating Cash Flow Metrics...")
        self.calculate_cashflow_metrics()
        
        print("7. Calculating Volatility Metrics...")
        self.calculate_volatility_metrics()
        
        print("8. Calculating Overall Credit Score...")
        self.calculate_credit_score()
        
        return self.results
    
    def print_summary(self):
        """Print formatted summary of results"""
        if not self.results:
            print("No results available. Run analysis first.")
            return
        
        print("\n" + "="*60)
        print("CREDIT ANALYSIS SUMMARY")
        print("="*60)
        
        # Credit Score
        if 'credit_score' in self.results:
            print(f"\n📊 OVERALL CREDIT SCORE: {self.results['credit_score']['score']:.1f}/100")
            print(f"   Rating: {self.results['credit_score']['rating']}")
        
        # Altman Z-Score
        if 'altman_z_score' in self.results:
            print(f"\n📈 ALTMAN Z-SCORE: {self.results['altman_z_score']['score']:.2f}")
            print(f"   {self.results['altman_z_score']['interpretation']}")
        
        # VaR/CVaR
        if 'var_cvar' in self.results:
            var = self.results['var_cvar']
            print(f"\n⚠️  VALUE AT RISK (95% confidence):")
            print(f"   Daily VaR: {abs(var['var_daily'])*100:.2f}%")
            print(f"   Annual VaR: {abs(var['var_annual'])*100:.2f}%")
            print(f"   Daily CVaR: {abs(var['cvar_daily'])*100:.2f}%")
        
        # Liquidity
        if 'liquidity_ratios' in self.results:
            liq = self.results['liquidity_ratios']
            print(f"\n💧 LIQUIDITY RATIOS:")
            print(f"   Current Ratio: {liq['current_ratio']:.2f} ({liq['interpretation']['current_ratio']})")
            print(f"   Quick Ratio: {liq['quick_ratio']:.2f} ({liq['interpretation']['quick_ratio']})")
        
        # Leverage
        if 'leverage_ratios' in self.results:
            lev = self.results['leverage_ratios']
            print(f"\n⚖️  LEVERAGE RATIOS:")
            print(f"   Debt/Equity: {lev['debt_to_equity']:.2f} ({lev['interpretation']['debt_to_equity']})")
            print(f"   Interest Coverage: {lev['interest_coverage']:.2f}x ({lev['interpretation']['interest_coverage']})")
        
        # Profitability
        if 'profitability_ratios' in self.results:
            prof = self.results['profitability_ratios']
            print(f"\n💰 PROFITABILITY:")
            print(f"   Net Margin: {prof['net_margin']*100:.2f}% ({prof['interpretation']['net_margin']})")
            print(f"   ROE: {prof['roe']*100:.2f}% ({prof['interpretation']['roe']})")
        
        print("\n" + "="*60)
    
    def export_to_dataframe(self):
        """Export results to pandas DataFrame"""
        data = []
        
        for category, metrics in self.results.items():
            if isinstance(metrics, dict):
                for key, value in metrics.items():
                    if key != 'interpretation' and key != 'components':
                        data.append({
                            'Category': category,
                            'Metric': key,
                            'Value': value
                        })
        
        return pd.DataFrame(data)


# Example Usage
if __name__ == "__main__":
    # Initialize model with a ticker
    ticker = "AAPL"  # Change this to any company ticker
    
    model = CreditWorthinessModel(ticker)
    
    # Run full analysis
    results = model.run_full_analysis()
    
    # Print summary
    model.print_summary()
    
    # Export to DataFrame
    df = model.export_to_dataframe()
    print("\n\nDetailed Metrics DataFrame:")
    print(df)
    
    # Access specific results
    print("\n\nAccess specific metrics:")
    print(f"Altman Z-Score: {results['altman_z_score']['score']:.2f}")
    print(f"Credit Rating: {results['credit_score']['rating']}")

Fetching data for AAPL...


429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/AAPL?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=AAPL&crumb=Edge%3A+Too+Many+Requests


Error fetching data: Expecting value: line 1 column 1 (char 0)
No results available. Run analysis first.


Detailed Metrics DataFrame:
Empty DataFrame
Columns: []
Index: []


Access specific metrics:


TypeError: 'NoneType' object is not subscriptable

In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy import stats
from datetime import datetime, timedelta
import warnings
import time
warnings.filterwarnings('ignore')

class CreditWorthinessModel:
    """
    Comprehensive credit worthiness analysis model using Yahoo Finance data
    """
    
    def __init__(self, ticker):
        self.ticker = ticker.upper()
        self.stock = None
        self.financials = None
        self.balance_sheet = None
        self.cashflow = None
        self.info = None
        self.historical_prices = None
        self.results = {}
        
    def fetch_data(self, max_retries=3):
        """Fetch all necessary financial data with retry logic"""
        print(f"Fetching data for {self.ticker}...")
        
        for attempt in range(max_retries):
            try:
                # Add delay to avoid rate limiting
                if attempt > 0:
                    wait_time = 2 ** attempt  # Exponential backoff
                    print(f"Retry attempt {attempt + 1}/{max_retries}, waiting {wait_time}s...")
                    time.sleep(wait_time)
                
                # Initialize ticker object
                self.stock = yf.Ticker(self.ticker)
                
                # Get financial statements with timeout
                print("  - Fetching financial statements...")
                self.financials = self.stock.financials
                self.balance_sheet = self.stock.balance_sheet
                self.cashflow = self.stock.cashflow
                
                # Get info separately with error handling
                print("  - Fetching company info...")
                try:
                    self.info = self.stock.info
                except:
                    # If info fails, create minimal info dict
                    self.info = {'marketCap': None, 'beta': None}
                
                # Get 5 years of historical price data
                print("  - Fetching historical prices...")
                end_date = datetime.now()
                start_date = end_date - timedelta(days=1*365)
                self.historical_prices = self.stock.history(start=start_date, end=end_date)
                
                # Validate that we got data
                if self.financials is None or self.financials.empty:
                    raise ValueError("No financial data available")
                if self.balance_sheet is None or self.balance_sheet.empty:
                    raise ValueError("No balance sheet data available")
                if self.historical_prices is None or len(self.historical_prices) < 100:
                    raise ValueError("Insufficient historical price data")
                
                print("✓ Data fetched successfully!")
                return True
                
            except Exception as e:
                print(f"✗ Attempt {attempt + 1} failed: {e}")
                if attempt == max_retries - 1:
                    print(f"\n❌ Failed to fetch data after {max_retries} attempts.")
                    print("Possible solutions:")
                    print("  1. Wait a few minutes and try again (rate limit)")
                    print("  2. Check if the ticker symbol is correct")
                    print("  3. Try a different company ticker")
                    return False
        
        return False
    
    def safe_get(self, dataframe, key, default=0):
        """Safely get value from dataframe"""
        try:
            if key in dataframe.index:
                val = dataframe[key]
                return val if not pd.isna(val) else default
            return default
        except:
            return default
    
    def calculate_altman_z_score(self):
        """
        Calculate Altman Z-Score for credit risk assessment
        Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
        """
        try:
            bs = self.balance_sheet.iloc[:, 0]
            income = self.financials.iloc[:, 0]
            
            # Extract values with safe get
            total_assets = self.safe_get(bs, 'Total Assets')
            current_assets = self.safe_get(bs, 'Current Assets')
            current_liabilities = self.safe_get(bs, 'Current Liabilities')
            retained_earnings = self.safe_get(bs, 'Retained Earnings')
            
            # Try multiple keys for EBIT
            ebit = self.safe_get(income, 'EBIT')
            if ebit == 0:
                ebit = self.safe_get(income, 'Operating Income')
            if ebit == 0:
                ebit = self.safe_get(income, 'EBITDA')
            
            total_liabilities = self.safe_get(bs, 'Total Liabilities Net Minority Interest')
            if total_liabilities == 0:
                total_liabilities = self.safe_get(bs, 'Total Liabilities')
            
            revenue = self.safe_get(income, 'Total Revenue')
            
            # Market cap
            market_cap = self.info.get('marketCap', 0) if self.info else 0
            
            if total_assets == 0:
                return None, "Insufficient data: Total Assets is zero"
            
            # Calculate components
            X1 = (current_assets - current_liabilities) / total_assets
            X2 = retained_earnings / total_assets
            X3 = ebit / total_assets
            X4 = market_cap / total_liabilities if total_liabilities > 0 else 0
            X5 = revenue / total_assets
            
            # Calculate Z-Score
            z_score = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
            
            # Interpretation
            if z_score > 2.99:
                interpretation = "Safe Zone (Low bankruptcy risk)"
            elif z_score > 1.81:
                interpretation = "Grey Zone (Moderate risk)"
            else:
                interpretation = "Distress Zone (High bankruptcy risk)"
            
            self.results['altman_z_score'] = {
                'score': z_score,
                'interpretation': interpretation,
                'components': {
                    'X1_working_capital_ratio': X1,
                    'X2_retained_earnings_ratio': X2,
                    'X3_ebit_ratio': X3,
                    'X4_market_equity_ratio': X4,
                    'X5_asset_turnover': X5
                }
            }
            
            return z_score, interpretation
            
        except Exception as e:
            print(f"  Warning: Could not calculate Altman Z-Score: {e}")
            return None, f"Error: {e}"
    
    def calculate_var_cvar(self, confidence_level=0.95):
        """Calculate Value at Risk (VaR) and Conditional VaR (CVaR)"""
        try:
            if self.historical_prices is None or len(self.historical_prices) < 2:
                return None, None, "Insufficient price data"
            
            # Calculate daily returns
            returns = self.historical_prices['Close'].pct_change().dropna()
            
            if len(returns) < 50:
                return None, None, "Insufficient return data"
            
            # Calculate VaR
            var = np.percentile(returns, (1 - confidence_level) * 100)
            
            # Calculate CVaR (Expected Shortfall)
            cvar = returns[returns <= var].mean()
            
            # Annualized metrics
            var_annual = var * np.sqrt(252)
            cvar_annual = cvar * np.sqrt(252)
            
            self.results['var_cvar'] = {
                'var_daily': var,
                'cvar_daily': cvar,
                'var_annual': var_annual,
                'cvar_annual': cvar_annual,
                'confidence_level': confidence_level,
                'interpretation': f"There is a {confidence_level*100}% chance that daily losses will not exceed {abs(var)*100:.2f}%"
            }
            
            return var, cvar, "Success"
            
        except Exception as e:
            print(f"  Warning: Could not calculate VaR/CVaR: {e}")
            return None, None, f"Error: {e}"
    
    def calculate_liquidity_ratios(self):
        """Calculate various liquidity ratios"""
        try:
            bs = self.balance_sheet.iloc[:, 0]
            
            current_assets = self.safe_get(bs, 'Current Assets')
            current_liabilities = self.safe_get(bs, 'Current Liabilities')
            cash = self.safe_get(bs, 'Cash And Cash Equivalents')
            inventory = self.safe_get(bs, 'Inventory')
            
            current_ratio = current_assets / current_liabilities if current_liabilities > 0 else 0
            quick_ratio = (current_assets - inventory) / current_liabilities if current_liabilities > 0 else 0
            cash_ratio = cash / current_liabilities if current_liabilities > 0 else 0
            
            self.results['liquidity_ratios'] = {
                'current_ratio': current_ratio,
                'quick_ratio': quick_ratio,
                'cash_ratio': cash_ratio,
                'interpretation': {
                    'current_ratio': 'Good' if current_ratio > 1.5 else 'Moderate' if current_ratio > 1 else 'Poor',
                    'quick_ratio': 'Good' if quick_ratio > 1 else 'Moderate' if quick_ratio > 0.5 else 'Poor'
                }
            }
            
            return self.results['liquidity_ratios']
            
        except Exception as e:
            print(f"  Warning: Could not calculate liquidity ratios: {e}")
            return None
    
    def calculate_leverage_ratios(self):
        """Calculate leverage and solvency ratios"""
        try:
            bs = self.balance_sheet.iloc[:, 0]
            income = self.financials.iloc[:, 0]
            
            total_debt = self.safe_get(bs, 'Total Debt')
            total_equity = self.safe_get(bs, 'Total Equity Gross Minority Interest')
            if total_equity == 0:
                total_equity = self.safe_get(bs, 'Stockholders Equity')
            
            total_assets = self.safe_get(bs, 'Total Assets')
            
            ebit = self.safe_get(income, 'EBIT')
            if ebit == 0:
                ebit = self.safe_get(income, 'Operating Income')
            
            interest_expense = self.safe_get(income, 'Interest Expense')
            
            debt_to_equity = total_debt / total_equity if total_equity > 0 else 0
            debt_to_assets = total_debt / total_assets if total_assets > 0 else 0
            equity_multiplier = total_assets / total_equity if total_equity > 0 else 0
            interest_coverage = ebit / abs(interest_expense) if interest_expense != 0 else 0
            
            self.results['leverage_ratios'] = {
                'debt_to_equity': debt_to_equity,
                'debt_to_assets': debt_to_assets,
                'equity_multiplier': equity_multiplier,
                'interest_coverage': interest_coverage,
                'interpretation': {
                    'debt_to_equity': 'Low leverage' if debt_to_equity < 1 else 'Moderate' if debt_to_equity < 2 else 'High leverage',
                    'interest_coverage': 'Strong' if interest_coverage > 5 else 'Adequate' if interest_coverage > 2.5 else 'Weak'
                }
            }
            
            return self.results['leverage_ratios']
            
        except Exception as e:
            print(f"  Warning: Could not calculate leverage ratios: {e}")
            return None
    
    def calculate_profitability_ratios(self):
        """Calculate profitability metrics"""
        try:
            income = self.financials.iloc[:, 0]
            bs = self.balance_sheet.iloc[:, 0]
            
            revenue = self.safe_get(income, 'Total Revenue')
            gross_profit = self.safe_get(income, 'Gross Profit')
            operating_income = self.safe_get(income, 'Operating Income')
            net_income = self.safe_get(income, 'Net Income')
            total_assets = self.safe_get(bs, 'Total Assets')
            total_equity = self.safe_get(bs, 'Total Equity Gross Minority Interest')
            if total_equity == 0:
                total_equity = self.safe_get(bs, 'Stockholders Equity')
            
            gross_margin = gross_profit / revenue if revenue > 0 else 0
            operating_margin = operating_income / revenue if revenue > 0 else 0
            net_margin = net_income / revenue if revenue > 0 else 0
            roa = net_income / total_assets if total_assets > 0 else 0
            roe = net_income / total_equity if total_equity > 0 else 0
            
            self.results['profitability_ratios'] = {
                'gross_margin': gross_margin,
                'operating_margin': operating_margin,
                'net_margin': net_margin,
                'roa': roa,
                'roe': roe,
                'interpretation': {
                    'net_margin': 'Strong' if net_margin > 0.15 else 'Moderate' if net_margin > 0.05 else 'Weak',
                    'roe': 'Excellent' if roe > 0.15 else 'Good' if roe > 0.10 else 'Below average'
                }
            }
            
            return self.results['profitability_ratios']
            
        except Exception as e:
            print(f"  Warning: Could not calculate profitability ratios: {e}")
            return None
    
    def calculate_cashflow_metrics(self):
        """Calculate cash flow related metrics"""
        try:
            cf = self.cashflow.iloc[:, 0]
            income = self.financials.iloc[:, 0]
            bs = self.balance_sheet.iloc[:, 0]
            
            operating_cf = self.safe_get(cf, 'Operating Cash Flow')
            free_cf = self.safe_get(cf, 'Free Cash Flow')
            net_income = self.safe_get(income, 'Net Income')
            total_debt = self.safe_get(bs, 'Total Debt')
            
            ocf_to_net_income = operating_cf / net_income if net_income > 0 else 0
            fcf_to_debt = free_cf / total_debt if total_debt > 0 else 0
            
            self.results['cashflow_metrics'] = {
                'operating_cashflow': operating_cf,
                'free_cashflow': free_cf,
                'ocf_to_net_income': ocf_to_net_income,
                'fcf_to_debt': fcf_to_debt,
                'interpretation': {
                    'ocf_quality': 'Strong' if ocf_to_net_income > 1 else 'Moderate' if ocf_to_net_income > 0.8 else 'Weak',
                    'debt_coverage': 'Strong' if fcf_to_debt > 0.2 else 'Moderate' if fcf_to_debt > 0.1 else 'Weak'
                }
            }
            
            return self.results['cashflow_metrics']
            
        except Exception as e:
            print(f"  Warning: Could not calculate cashflow metrics: {e}")
            return None
    
    def calculate_volatility_metrics(self):
        """Calculate stock volatility and beta"""
        try:
            returns = self.historical_prices['Close'].pct_change().dropna()
            
            # Historical volatility
            daily_vol = returns.std()
            annual_vol = daily_vol * np.sqrt(252)
            
            # Beta from Yahoo Finance
            beta = self.info.get('beta', None) if self.info else None
            
            # Sharpe Ratio (assuming risk-free rate of 4%)
            risk_free_rate = 0.04
            avg_return = returns.mean() * 252
            sharpe = (avg_return - risk_free_rate) / annual_vol if annual_vol > 0 else 0
            
            self.results['volatility_metrics'] = {
                'daily_volatility': daily_vol,
                'annual_volatility': annual_vol,
                'beta': beta,
                'sharpe_ratio': sharpe,
                'interpretation': {
                    'volatility': 'Low' if annual_vol < 0.2 else 'Moderate' if annual_vol < 0.4 else 'High',
                    'beta': 'Defensive' if beta and beta < 1 else 'Market' if beta and beta < 1.2 else 'Aggressive' if beta else 'N/A'
                }
            }
            
            return self.results['volatility_metrics']
            
        except Exception as e:
            print(f"  Warning: Could not calculate volatility metrics: {e}")
            return None
    
    def calculate_credit_score(self):
        """Calculate overall credit score (0-100)"""
        try:
            score = 50  # Base score
            
            # Altman Z-Score component (0-20 points)
            if 'altman_z_score' in self.results:
                z = self.results['altman_z_score']['score']
                if z > 2.99:
                    score += 20
                elif z > 1.81:
                    score += 10
                else:
                    score += 0
            
            # Liquidity component (0-15 points)
            if 'liquidity_ratios' in self.results:
                current_ratio = self.results['liquidity_ratios']['current_ratio']
                if current_ratio > 2:
                    score += 15
                elif current_ratio > 1:
                    score += 10
                else:
                    score += 5
            
            # Leverage component (0-20 points)
            if 'leverage_ratios' in self.results:
                d_to_e = self.results['leverage_ratios']['debt_to_equity']
                int_cov = self.results['leverage_ratios']['interest_coverage']
                
                if d_to_e < 1:
                    score += 10
                elif d_to_e < 2:
                    score += 5
                
                if int_cov > 5:
                    score += 10
                elif int_cov > 2.5:
                    score += 5
            
            # Profitability component (0-15 points)
            if 'profitability_ratios' in self.results:
                roe = self.results['profitability_ratios']['roe']
                net_margin = self.results['profitability_ratios']['net_margin']
                
                if roe > 0.15:
                    score += 8
                elif roe > 0.10:
                    score += 5
                
                if net_margin > 0.15:
                    score += 7
                elif net_margin > 0.05:
                    score += 4
            
            # Risk component (0-15 points)
            if 'var_cvar' in self.results:
                var_annual = abs(self.results['var_cvar']['var_annual'])
                if var_annual < 0.15:
                    score += 15
                elif var_annual < 0.25:
                    score += 10
                else:
                    score += 5
            
            # Cash flow component (0-15 points)
            if 'cashflow_metrics' in self.results:
                ocf_ratio = self.results['cashflow_metrics']['ocf_to_net_income']
                if ocf_ratio > 1:
                    score += 15
                elif ocf_ratio > 0.8:
                    score += 10
                else:
                    score += 5
            
            score = min(score, 100)
            
            # Rating
            if score >= 85:
                rating = "AAA - Excellent"
            elif score >= 75:
                rating = "AA - Very Good"
            elif score >= 65:
                rating = "A - Good"
            elif score >= 55:
                rating = "BBB - Adequate"
            elif score >= 45:
                rating = "BB - Moderate Risk"
            elif score >= 35:
                rating = "B - High Risk"
            else:
                rating = "CCC - Very High Risk"
            
            self.results['credit_score'] = {
                'score': score,
                'rating': rating
            }
            
            return score, rating
            
        except Exception as e:
            print(f"  Warning: Could not calculate credit score: {e}")
            return None, f"Error: {e}"
    
    def run_full_analysis(self):
        """Run complete credit analysis"""
        if not self.fetch_data():
            print("\n❌ Analysis aborted due to data fetch failure.")
            return None
        
        print("\n" + "="*60)
        print(f"CREDIT WORTHINESS ANALYSIS: {self.ticker}")
        print("="*60)
        
        # Run all calculations
        print("\n1. Calculating Altman Z-Score...")
        self.calculate_altman_z_score()
        
        print("2. Calculating VaR and CVaR...")
        self.calculate_var_cvar()
        
        print("3. Calculating Liquidity Ratios...")
        self.calculate_liquidity_ratios()
        
        print("4. Calculating Leverage Ratios...")
        self.calculate_leverage_ratios()
        
        print("5. Calculating Profitability Ratios...")
        self.calculate_profitability_ratios()
        
        print("6. Calculating Cash Flow Metrics...")
        self.calculate_cashflow_metrics()
        
        print("7. Calculating Volatility Metrics...")
        self.calculate_volatility_metrics()
        
        print("8. Calculating Overall Credit Score...")
        self.calculate_credit_score()
        
        print("\n✓ Analysis complete!")
        return self.results
    
    def print_summary(self):
        """Print formatted summary of results"""
        if not self.results:
            print("No results available. Run analysis first.")
            return
        
        print("\n" + "="*60)
        print("CREDIT ANALYSIS SUMMARY")
        print("="*60)
        
        # Credit Score
        if 'credit_score' in self.results:
            print(f"\n📊 OVERALL CREDIT SCORE: {self.results['credit_score']['score']:.1f}/100")
            print(f"   Rating: {self.results['credit_score']['rating']}")
        
        # Altman Z-Score
        if 'altman_z_score' in self.results:
            print(f"\n📈 ALTMAN Z-SCORE: {self.results['altman_z_score']['score']:.2f}")
            print(f"   {self.results['altman_z_score']['interpretation']}")
        
        # VaR/CVaR
        if 'var_cvar' in self.results:
            var = self.results['var_cvar']
            print(f"\n⚠️  VALUE AT RISK (95% confidence):")
            print(f"   Daily VaR: {abs(var['var_daily'])*100:.2f}%")
            print(f"   Annual VaR: {abs(var['var_annual'])*100:.2f}%")
            print(f"   Daily CVaR: {abs(var['cvar_daily'])*100:.2f}%")
        
        # Liquidity
        if 'liquidity_ratios' in self.results:
            liq = self.results['liquidity_ratios']
            print(f"\n💧 LIQUIDITY RATIOS:")
            print(f"   Current Ratio: {liq['current_ratio']:.2f} ({liq['interpretation']['current_ratio']})")
            print(f"   Quick Ratio: {liq['quick_ratio']:.2f} ({liq['interpretation']['quick_ratio']})")
        
        # Leverage
        if 'leverage_ratios' in self.results:
            lev = self.results['leverage_ratios']
            print(f"\n⚖️  LEVERAGE RATIOS:")
            print(f"   Debt/Equity: {lev['debt_to_equity']:.2f} ({lev['interpretation']['debt_to_equity']})")
            print(f"   Interest Coverage: {lev['interest_coverage']:.2f}x ({lev['interpretation']['interest_coverage']})")
        
        # Profitability
        if 'profitability_ratios' in self.results:
            prof = self.results['profitability_ratios']
            print(f"\n💰 PROFITABILITY:")
            print(f"   Net Margin: {prof['net_margin']*100:.2f}% ({prof['interpretation']['net_margin']})")
            print(f"   ROE: {prof['roe']*100:.2f}% ({prof['interpretation']['roe']})")
        
        print("\n" + "="*60)
    
    def export_to_dataframe(self):
        """Export results to pandas DataFrame"""
        data = []
        
        for category, metrics in self.results.items():
            if isinstance(metrics, dict):
                for key, value in metrics.items():
                    if key != 'interpretation' and key != 'components' and not isinstance(value, dict):
                        data.append({
                            'Category': category,
                            'Metric': key,
                            'Value': value
                        })
        
        return pd.DataFrame(data)


# Example Usage
if __name__ == "__main__":
    # Initialize model with a ticker
    ticker = "MSFT"  # Change this to any company ticker
    
    print("="*60)
    print("CREDIT WORTHINESS MODEL")
    print("="*60)
    print(f"Analyzing: {ticker}")
    print("\nNote: This may take 30-60 seconds due to API rate limits...")
    
    model = CreditWorthinessModel(ticker)
    
    # Run full analysis
    results = model.run_full_analysis()
    
    # Only proceed if we have results
    if results:
        # Print summary
        model.print_summary()
        
        # Export to DataFrame
        df = model.export_to_dataframe()
        print("\n\nDetailed Metrics DataFrame:")
        print(df.to_string(index=False))
        
        # Access specific results safely
        if 'altman_z_score' in results and 'credit_score' in results:
            print("\n\nKey Metrics:")
            print(f"  Altman Z-Score: {results['altman_z_score']['score']:.2f}")
            print(f"  Credit Rating: {results['credit_score']['rating']}")
    else:
        print("\n⚠️  Unable to complete analysis. Please try again in a few minutes.")

CREDIT WORTHINESS MODEL
Analyzing: MSFT

Note: This may take 30-60 seconds due to API rate limits...
Fetching data for MSFT...
  - Fetching financial statements...
  - Fetching company info...


429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/MSFT?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=MSFT&crumb=Edge%3A+Too+Many+Requests
Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)
$MSFT: possibly delisted; no timezone found


  - Fetching historical prices...
✗ Attempt 1 failed: No financial data available
Retry attempt 2/3, waiting 2s...
  - Fetching financial statements...
  - Fetching company info...


429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/MSFT?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=MSFT&crumb=Edge%3A+Too+Many+Requests
Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)
$MSFT: possibly delisted; no timezone found


  - Fetching historical prices...
✗ Attempt 2 failed: No financial data available
Retry attempt 3/3, waiting 4s...
  - Fetching financial statements...
  - Fetching company info...


429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/MSFT?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=MSFT&crumb=Edge%3A+Too+Many+Requests
Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)
$MSFT: possibly delisted; no timezone found


  - Fetching historical prices...
✗ Attempt 3 failed: No financial data available

❌ Failed to fetch data after 3 attempts.
Possible solutions:
  1. Wait a few minutes and try again (rate limit)
  2. Check if the ticker symbol is correct
  3. Try a different company ticker

❌ Analysis aborted due to data fetch failure.

⚠️  Unable to complete analysis. Please try again in a few minutes.


In [15]:
import requests

# Test your API key
api_key = "ulHZsJKoEkf8lBZRjP3saC8SWwUP9777"
ticker = "AAPL"
url = f"https://financialmodelingprep.com/api/v3/profile/{ticker}?apikey={api_key}"

response = requests.get(url)
if response.status_code == 200:
    print("✓ API Key works!")
    print(response.json())
else:
    print("❌ API Key issue")
    print(response.text)

❌ API Key issue
{
  "Error Message": "Legacy Endpoint : Due to Legacy endpoints being no longer supported - This endpoint is only available for legacy users who have valid subscriptions prior August 31, 2025. Please visit our subscription page to upgrade your plan or contact us at https://site.financialmodelingprep.com/developer/docs/pricing"
}


In [17]:
"""
CREDIT WORTHINESS MODEL - 100% FREE VERSION
Uses: SEC Edgar (financials) + yfinance (prices with fallbacks)
No API key needed!

Install: pip install sec-edgar-downloader pandas numpy scipy yfinance requests beautifulsoup4 lxml
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import time
import requests
from io import StringIO
warnings.filterwarnings('ignore')

class CreditWorthinessModel:
    """
    Free credit worthiness analysis using SEC Edgar + multiple price sources
    """
    
    def __init__(self, ticker):
        self.ticker = ticker.upper()
        self.cik = None
        self.financials = {}
        self.historical_prices = None
        self.results = {}
        
    def get_cik(self):
        """Get CIK number from ticker (needed for SEC)"""
        try:
            # SEC company tickers JSON
            url = "https://www.sec.gov/files/company_tickers.json"
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                for item in data.values():
                    if item['ticker'].upper() == self.ticker:
                        self.cik = str(item['cik_str']).zfill(10)
                        print(f"  ✓ Found CIK: {self.cik}")
                        return True
                print(f"  ✗ Ticker {self.ticker} not found in SEC database")
                return False
        except Exception as e:
            print(f"  ✗ Error getting CIK: {e}")
            return False
    
    def fetch_price_data_yfinance(self):
        """Try to fetch prices from yfinance"""
        try:
            import yfinance as yf
            print("  - Trying yfinance...")
            
            end_date = datetime.now()
            start_date = end_date - timedelta(days=5*365)
            
            stock = yf.Ticker(self.ticker)
            df = stock.history(start=start_date, end=end_date, auto_adjust=False)
            
            if df is not None and len(df) > 100:
                self.historical_prices = df
                print(f"    ✓ Got {len(df)} days from yfinance")
                return True
            return False
        except Exception as e:
            print(f"    ✗ yfinance failed: {e}")
            return False
    
    def fetch_price_data_yahoo_direct(self):
        """Direct Yahoo Finance API (no library)"""
        try:
            print("  - Trying Yahoo Finance direct API...")
            
            end = int(datetime.now().timestamp())
            start = int((datetime.now() - timedelta(days=5*365)).timestamp())
            
            url = f"https://query1.finance.yahoo.com/v7/finance/download/{self.ticker}"
            params = {
                'period1': start,
                'period2': end,
                'interval': '1d',
                'events': 'history'
            }
            headers = {'User-Agent': 'Mozilla/5.0'}
            
            time.sleep(1)  # Be nice to Yahoo
            response = requests.get(url, params=params, headers=headers, timeout=10)
            
            if response.status_code == 200:
                df = pd.read_csv(StringIO(response.text))
                df['Date'] = pd.to_datetime(df['Date'])
                df.set_index('Date', inplace=True)
                
                if len(df) > 100:
                    # Rename columns to match yfinance format
                    df.rename(columns={'Adj Close': 'Close'}, inplace=True)
                    self.historical_prices = df
                    print(f"    ✓ Got {len(df)} days from Yahoo direct")
                    return True
            return False
        except Exception as e:
            print(f"    ✗ Yahoo direct failed: {e}")
            return False
    
    def fetch_price_data_alphavantage_free(self):
        """Try Alpha Vantage demo/free endpoint"""
        try:
            print("  - Trying Alpha Vantage free endpoint...")
            
            # Alpha Vantage has a demo key that works for limited requests
            url = "https://www.alphavantage.co/query"
            params = {
                'function': 'TIME_SERIES_DAILY',
                'symbol': self.ticker,
                'outputsize': 'full',
                'apikey': 'demo'  # Limited demo key
            }
            
            time.sleep(2)
            response = requests.get(url, params=params, timeout=15)
            
            if response.status_code == 200:
                data = response.json()
                if 'Time Series (Daily)' in data:
                    prices = data['Time Series (Daily)']
                    df = pd.DataFrame.from_dict(prices, orient='index')
                    df.index = pd.to_datetime(df.index)
                    df = df.sort_index()
                    df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
                    
                    for col in df.columns:
                        df[col] = pd.to_numeric(df[col])
                    
                    cutoff = datetime.now() - timedelta(days=5*365)
                    df = df[df.index >= cutoff]
                    
                    if len(df) > 100:
                        self.historical_prices = df
                        print(f"    ✓ Got {len(df)} days from Alpha Vantage")
                        return True
            return False
        except Exception as e:
            print(f"    ✗ Alpha Vantage failed: {e}")
            return False
    
    def fetch_sec_financials(self):
        """Fetch financial data from SEC Edgar"""
        try:
            if not self.cik:
                if not self.get_cik():
                    return False
            
            print(f"  - Fetching SEC filings for CIK {self.cik}...")
            
            # SEC company facts API (XBRL data)
            url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{self.cik}.json"
            headers = {
                'User-Agent': 'YourName your.email@example.com',  # SEC requires user agent
                'Accept-Encoding': 'gzip, deflate'
            }
            
            time.sleep(0.5)  # Be respectful to SEC servers
            response = requests.get(url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                print(f"    ✗ SEC API returned status {response.status_code}")
                return False
            
            data = response.json()
            
            # Extract financial data
            if 'facts' in data:
                us_gaap = data['facts'].get('us-gaap', {})
                
                # Get the most recent annual filings (10-K)
                def get_recent_values(concept, num_years=5):
                    if concept not in us_gaap:
                        return []
                    
                    units = us_gaap[concept].get('units', {})
                    # Usually in USD
                    usd_data = units.get('USD', [])
                    
                    # Filter for annual reports (10-K)
                    annual = [x for x in usd_data if x.get('form') == '10-K']
                    # Sort by date descending
                    annual.sort(key=lambda x: x.get('end', ''), reverse=True)
                    
                    return annual[:num_years]
                
                # Key financial metrics
                self.financials['Assets'] = get_recent_values('Assets')
                self.financials['AssetsCurrent'] = get_recent_values('AssetsCurrent')
                self.financials['LiabilitiesCurrent'] = get_recent_values('LiabilitiesCurrent')
                self.financials['Liabilities'] = get_recent_values('Liabilities')
                self.financials['StockholdersEquity'] = get_recent_values('StockholdersEquity')
                self.financials['RetainedEarningsAccumulatedDeficit'] = get_recent_values('RetainedEarningsAccumulatedDeficit')
                self.financials['Revenues'] = get_recent_values('Revenues')
                self.financials['OperatingIncomeLoss'] = get_recent_values('OperatingIncomeLoss')
                self.financials['NetIncomeLoss'] = get_recent_values('NetIncomeLoss')
                self.financials['OperatingCashFlow'] = get_recent_values('NetCashProvidedByUsedInOperatingActivities')
                self.financials['DebtCurrent'] = get_recent_values('LongTermDebtCurrent')
                self.financials['DebtNoncurrent'] = get_recent_values('LongTermDebtNoncurrent')
                self.financials['InterestExpense'] = get_recent_values('InterestExpense')
                self.financials['CashAndEquivalents'] = get_recent_values('CashAndCashEquivalentsAtCarryingValue')
                self.financials['Inventory'] = get_recent_values('InventoryNet')
                
                print(f"    ✓ Retrieved SEC financial data")
                return True
            else:
                print("    ✗ No financial facts found in SEC data")
                return False
                
        except Exception as e:
            print(f"    ✗ Error fetching SEC data: {e}")
            return False
    
    def fetch_data(self):
        """Main data fetching with multiple fallbacks"""
        print(f"\nFetching data for {self.ticker}...")
        print("="*60)
        
        # 1. Get SEC financials
        print("\n1. Fetching Financial Statements (SEC Edgar - Free):")
        if not self.fetch_sec_financials():
            print("\n❌ Could not fetch SEC data. This ticker may not file with SEC.")
            print("   Note: Only US public companies file with SEC")
            return False
        
        # 2. Get price data with fallbacks
        print("\n2. Fetching Historical Prices (trying multiple sources):")
        
        success = (self.fetch_price_data_yfinance() or 
                  self.fetch_price_data_yahoo_direct() or
                  self.fetch_price_data_alphavantage_free())
        
        if not success:
            print("\n⚠️  Could not fetch price data from any source")
            print("   Will calculate metrics without price-based analysis")
            print("   (VaR, CVaR, and volatility metrics will be unavailable)")
        
        print("\n" + "="*60)
        print("✓ Data collection complete!")
        return True
    
    def get_latest_value(self, key, default=0):
        """Get the most recent value for a financial metric"""
        try:
            if key in self.financials and self.financials[key]:
                return float(self.financials[key][0].get('val', default))
            return default
        except:
            return default
    
    def calculate_altman_z_score(self):
        """Calculate Altman Z-Score"""
        try:
            # Get values from SEC data
            total_assets = self.get_latest_value('Assets')
            current_assets = self.get_latest_value('AssetsCurrent')
            current_liabilities = self.get_latest_value('LiabilitiesCurrent')
            retained_earnings = self.get_latest_value('RetainedEarningsAccumulatedDeficit')
            ebit = self.get_latest_value('OperatingIncomeLoss')
            total_liabilities = self.get_latest_value('Liabilities')
            revenue = self.get_latest_value('Revenues')
            stockholders_equity = self.get_latest_value('StockholdersEquity')
            
            if total_assets == 0:
                print("  ✗ Cannot calculate Altman Z-Score (missing data)")
                return None, "Insufficient data"
            
            # Calculate components
            X1 = (current_assets - current_liabilities) / total_assets
            X2 = retained_earnings / total_assets
            X3 = ebit / total_assets
            
            # For X4, use book value of equity since we may not have market cap
            X4 = stockholders_equity / total_liabilities if total_liabilities > 0 else 0
            X5 = revenue / total_assets
            
            # Altman Z-Score (using book value version)
            z_score = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
            
            # Interpretation
            if z_score > 2.99:
                interpretation = "Safe Zone (Low bankruptcy risk)"
            elif z_score > 1.81:
                interpretation = "Grey Zone (Moderate risk)"
            else:
                interpretation = "Distress Zone (High bankruptcy risk)"
            
            self.results['altman_z_score'] = {
                'score': z_score,
                'interpretation': interpretation,
                'components': {
                    'X1_working_capital_ratio': X1,
                    'X2_retained_earnings_ratio': X2,
                    'X3_ebit_ratio': X3,
                    'X4_equity_ratio': X4,
                    'X5_asset_turnover': X5
                }
            }
            
            print(f"  ✓ Altman Z-Score: {z_score:.2f} ({interpretation})")
            return z_score, interpretation
            
        except Exception as e:
            print(f"  ✗ Error calculating Altman Z-Score: {e}")
            return None, str(e)
    
    def calculate_var_cvar(self, confidence_level=0.95):
        """Calculate VaR and CVaR"""
        try:
            if self.historical_prices is None:
                print("  ⚠️  Skipping VaR/CVaR (no price data)")
                return None, None, "No price data"
            
            returns = self.historical_prices['Close'].pct_change().dropna()
            
            if len(returns) < 50:
                print("  ⚠️  Insufficient return data")
                return None, None, "Insufficient data"
            
            var = np.percentile(returns, (1 - confidence_level) * 100)
            cvar = returns[returns <= var].mean()
            
            var_annual = var * np.sqrt(252)
            cvar_annual = cvar * np.sqrt(252)
            
            self.results['var_cvar'] = {
                'var_daily': var,
                'cvar_daily': cvar,
                'var_annual': var_annual,
                'cvar_annual': cvar_annual,
                'confidence_level': confidence_level
            }
            
            print(f"  ✓ VaR (95%): {abs(var)*100:.2f}% daily, {abs(var_annual)*100:.2f}% annual")
            return var, cvar, "Success"
            
        except Exception as e:
            print(f"  ⚠️  VaR/CVaR calculation skipped: {e}")
            return None, None, str(e)
    
    def calculate_liquidity_ratios(self):
        """Calculate liquidity ratios"""
        try:
            current_assets = self.get_latest_value('AssetsCurrent')
            current_liabilities = self.get_latest_value('LiabilitiesCurrent')
            cash = self.get_latest_value('CashAndEquivalents')
            inventory = self.get_latest_value('Inventory')
            
            current_ratio = current_assets / current_liabilities if current_liabilities > 0 else 0
            quick_ratio = (current_assets - inventory) / current_liabilities if current_liabilities > 0 else 0
            cash_ratio = cash / current_liabilities if current_liabilities > 0 else 0
            
            self.results['liquidity_ratios'] = {
                'current_ratio': current_ratio,
                'quick_ratio': quick_ratio,
                'cash_ratio': cash_ratio
            }
            
            print(f"  ✓ Current Ratio: {current_ratio:.2f}, Quick Ratio: {quick_ratio:.2f}")
            return self.results['liquidity_ratios']
            
        except Exception as e:
            print(f"  ✗ Error calculating liquidity: {e}")
            return None
    
    def calculate_leverage_ratios(self):
        """Calculate leverage ratios"""
        try:
            debt_current = self.get_latest_value('DebtCurrent')
            debt_noncurrent = self.get_latest_value('DebtNoncurrent')
            total_debt = debt_current + debt_noncurrent
            
            total_equity = self.get_latest_value('StockholdersEquity')
            total_assets = self.get_latest_value('Assets')
            ebit = self.get_latest_value('OperatingIncomeLoss')
            interest_expense = self.get_latest_value('InterestExpense')
            
            debt_to_equity = total_debt / total_equity if total_equity > 0 else 0
            debt_to_assets = total_debt / total_assets if total_assets > 0 else 0
            interest_coverage = ebit / abs(interest_expense) if interest_expense != 0 else 0
            
            self.results['leverage_ratios'] = {
                'debt_to_equity': debt_to_equity,
                'debt_to_assets': debt_to_assets,
                'interest_coverage': interest_coverage
            }
            
            print(f"  ✓ Debt/Equity: {debt_to_equity:.2f}, Interest Coverage: {interest_coverage:.2f}x")
            return self.results['leverage_ratios']
            
        except Exception as e:
            print(f"  ✗ Error calculating leverage: {e}")
            return None
    
    def calculate_profitability_ratios(self):
        """Calculate profitability ratios"""
        try:
            revenue = self.get_latest_value('Revenues')
            operating_income = self.get_latest_value('OperatingIncomeLoss')
            net_income = self.get_latest_value('NetIncomeLoss')
            total_assets = self.get_latest_value('Assets')
            total_equity = self.get_latest_value('StockholdersEquity')
            
            operating_margin = operating_income / revenue if revenue > 0 else 0
            net_margin = net_income / revenue if revenue > 0 else 0
            roa = net_income / total_assets if total_assets > 0 else 0
            roe = net_income / total_equity if total_equity > 0 else 0
            
            self.results['profitability_ratios'] = {
                'operating_margin': operating_margin,
                'net_margin': net_margin,
                'roa': roa,
                'roe': roe
            }
            
            print(f"  ✓ Net Margin: {net_margin*100:.2f}%, ROE: {roe*100:.2f}%")
            return self.results['profitability_ratios']
            
        except Exception as e:
            print(f"  ✗ Error calculating profitability: {e}")
            return None
    
    def calculate_credit_score(self):
        """Calculate overall credit score"""
        try:
            score = 50
            
            # Altman Z-Score (0-20 pts)
            if 'altman_z_score' in self.results:
                z = self.results['altman_z_score']['score']
                score += 20 if z > 2.99 else 10 if z > 1.81 else 0
            
            # Liquidity (0-15 pts)
            if 'liquidity_ratios' in self.results:
                cr = self.results['liquidity_ratios']['current_ratio']
                score += 15 if cr > 2 else 10 if cr > 1 else 5
            
            # Leverage (0-20 pts)
            if 'leverage_ratios' in self.results:
                d_to_e = self.results['leverage_ratios']['debt_to_equity']
                int_cov = self.results['leverage_ratios']['interest_coverage']
                score += 10 if d_to_e < 1 else 5 if d_to_e < 2 else 0
                score += 10 if int_cov > 5 else 5 if int_cov > 2.5 else 0
            
            # Profitability (0-15 pts)
            if 'profitability_ratios' in self.results:
                roe = self.results['profitability_ratios']['roe']
                net_margin = self.results['profitability_ratios']['net_margin']
                score += 8 if roe > 0.15 else 5 if roe > 0.10 else 0
                score += 7 if net_margin > 0.15 else 4 if net_margin > 0.05 else 0
            
            score = min(score, 100)
            
            # Rating
            if score >= 85:
                rating = "AAA - Excellent"
            elif score >= 75:
                rating = "AA - Very Good"
            elif score >= 65:
                rating = "A - Good"
            elif score >= 55:
                rating = "BBB - Adequate"
            elif score >= 45:
                rating = "BB - Moderate Risk"
            else:
                rating = "B - High Risk"
            
            self.results['credit_score'] = {
                'score': score,
                'rating': rating
            }
            
            print(f"  ✓ Credit Score: {score:.0f}/100 ({rating})")
            return score, rating
            
        except Exception as e:
            print(f"  ✗ Error calculating credit score: {e}")
            return None, str(e)
    
    def run_full_analysis(self):
        """Run complete analysis"""
        if not self.fetch_data():
            return None
        
        print("\n" + "="*60)
        print(f"CALCULATING CREDIT METRICS: {self.ticker}")
        print("="*60 + "\n")
        
        print("1. Altman Z-Score")
        self.calculate_altman_z_score()
        
        print("\n2. VaR and CVaR (95% confidence)")
        self.calculate_var_cvar()
        
        print("\n3. Liquidity Ratios")
        self.calculate_liquidity_ratios()
        
        print("\n4. Leverage Ratios")
        self.calculate_leverage_ratios()
        
        print("\n5. Profitability Ratios")
        self.calculate_profitability_ratios()
        
        print("\n6. Overall Credit Score")
        self.calculate_credit_score()
        
        print("\n" + "="*60)
        print("✓ ANALYSIS COMPLETE")
        print("="*60)
        
        return self.results
    
    def print_summary(self):
        """Print detailed summary"""
        if not self.results:
            print("No results available")
            return
        
        print("\n" + "="*60)
        print(f"CREDIT ANALYSIS SUMMARY: {self.ticker}")
        print("="*60)
        
        if 'credit_score' in self.results:
            print(f"\n📊 OVERALL CREDIT SCORE: {self.results['credit_score']['score']:.0f}/100")
            print(f"   Rating: {self.results['credit_score']['rating']}")
        
        if 'altman_z_score' in self.results:
            z = self.results['altman_z_score']
            print(f"\n📈 ALTMAN Z-SCORE: {z['score']:.2f}")
            print(f"   {z['interpretation']}")
        
        if 'var_cvar' in self.results:
            var = self.results['var_cvar']
            print(f"\n⚠️  VALUE AT RISK (95% confidence):")
            print(f"   Daily VaR: {abs(var['var_daily'])*100:.2f}%")
            print(f"   Annual VaR: {abs(var['var_annual'])*100:.2f}%")
            print(f"   Daily CVaR: {abs(var['cvar_daily'])*100:.2f}%")
        
        if 'liquidity_ratios' in self.results:
            liq = self.results['liquidity_ratios']
            print(f"\n💧 LIQUIDITY:")
            print(f"   Current Ratio: {liq['current_ratio']:.2f}")
            print(f"   Quick Ratio: {liq['quick_ratio']:.2f}")
        
        if 'leverage_ratios' in self.results:
            lev = self.results['leverage_ratios']
            print(f"\n⚖️  LEVERAGE:")
            print(f"   Debt/Equity: {lev['debt_to_equity']:.2f}")
            print(f"   Interest Coverage: {lev['interest_coverage']:.2f}x")
        
        if 'profitability_ratios' in self.results:
            prof = self.results['profitability_ratios']
            print(f"\n💰 PROFITABILITY:")
            print(f"   Net Margin: {prof['net_margin']*100:.2f}%")
            print(f"   ROE: {prof['roe']*100:.2f}%")
        
        print("\n" + "="*60)


# ==============================================================================
# EXAMPLE USAGE - 100% FREE!
# ==============================================================================

if __name__ == "__main__":
    print("="*60)
    print("CREDIT WORTHINESS MODEL - 100% FREE VERSION")
    print("="*60)
    print("\nNo API key needed!")
    print("Uses: SEC Edgar (financials) + Multiple free price sources")
    print("\nNote: Only works for US companies that file with SEC")
    print("="*60)
    
    # Example - just change the ticker!
    ticker = "MSFT"  # Try: MSFT, GOOGL, JPM, TSLA, etc.
    
    model = CreditWorthinessModel(ticker)
    results = model.run_full_analysis()
    
    if results:
        model.print_summary()
        
        # Export to DataFrame
        data = []
        for category, metrics in results.items():
            if isinstance(metrics, dict) and category not in ['credit_score']:
                for key, value in metrics.items():
                    if key not in ['interpretation', 'components'] and not isinstance(value, dict):
                        data.append({
                            'Category': category.replace('_', ' ').title(),
                            'Metric': key.replace('_', ' ').title(),
                            'Value': f"{value:.4f}" if isinstance(value, float) else value
                        })
        
        if data:
            df = pd.DataFrame(data)
            print("\n" + "="*60)
            print("DETAILED METRICS")
            print("="*60)
            print(df.to_string(index=False))

CREDIT WORTHINESS MODEL - 100% FREE VERSION

No API key needed!
Uses: SEC Edgar (financials) + Multiple free price sources

Note: Only works for US companies that file with SEC

Fetching data for MSFT...

1. Fetching Financial Statements (SEC Edgar - Free):

❌ Could not fetch SEC data. This ticker may not file with SEC.
   Note: Only US public companies file with SEC


In [18]:
"""
CREDIT WORTHINESS MODEL - 100% FREE VERSION
Uses: SEC Edgar (financials) + yfinance (prices with fallbacks)
No API key needed!

Install: pip install sec-edgar-downloader pandas numpy scipy yfinance requests beautifulsoup4 lxml
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import time
import requests
from io import StringIO
warnings.filterwarnings('ignore')

class CreditWorthinessModel:
    """
    Free credit worthiness analysis using SEC Edgar + multiple price sources
    """
    
    def __init__(self, ticker):
        self.ticker = ticker.upper()
        self.cik = None
        self.financials = {}
        self.historical_prices = None
        self.results = {}
        
    def get_cik(self):
        """Get CIK number from ticker (needed for SEC)"""
        try:
            # SEC company tickers JSON
            url = "https://www.sec.gov/files/company_tickers.json"
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                for item in data.values():
                    if item['ticker'].upper() == self.ticker:
                        self.cik = str(item['cik_str']).zfill(10)
                        print(f"  ✓ Found CIK: {self.cik}")
                        return True
                print(f"  ✗ Ticker {self.ticker} not found in SEC database")
                return False
        except Exception as e:
            print(f"  ✗ Error getting CIK: {e}")
            return False
    
    def fetch_data_yfinance_complete(self):
        """Fetch both financials and prices from yfinance"""
        try:
            import yfinance as yf
            print("  - Attempting yfinance for financial statements...")
            
            stock = yf.Ticker(self.ticker)
            
            # Get financial statements
            time.sleep(1)
            bs = stock.balance_sheet
            income = stock.financials
            cf = stock.cashflow
            
            if bs is None or bs.empty or income is None or income.empty:
                print("    ✗ yfinance: No financial data")
                return False
            
            # Convert to our format
            def yf_get_latest(df, key, default=0):
                try:
                    if key in df.index:
                        val = df.loc[key].iloc[0] if hasattr(df.loc[key], 'iloc') else df.loc[key]
                        return float(val) if not pd.isna(val) else default
                    return default
                except:
                    return default
            
            # Map to our internal structure
            self.financials['Assets'] = [{'val': yf_get_latest(bs, 'Total Assets')}]
            self.financials['AssetsCurrent'] = [{'val': yf_get_latest(bs, 'Current Assets')}]
            self.financials['LiabilitiesCurrent'] = [{'val': yf_get_latest(bs, 'Current Liabilities')}]
            self.financials['Liabilities'] = [{'val': yf_get_latest(bs, 'Total Liabilities Net Minority Interest')}]
            self.financials['StockholdersEquity'] = [{'val': yf_get_latest(bs, 'Stockholders Equity')}]
            self.financials['RetainedEarningsAccumulatedDeficit'] = [{'val': yf_get_latest(bs, 'Retained Earnings')}]
            self.financials['Revenues'] = [{'val': yf_get_latest(income, 'Total Revenue')}]
            self.financials['OperatingIncomeLoss'] = [{'val': yf_get_latest(income, 'Operating Income')}]
            self.financials['NetIncomeLoss'] = [{'val': yf_get_latest(income, 'Net Income')}]
            self.financials['OperatingCashFlow'] = [{'val': yf_get_latest(cf, 'Operating Cash Flow')}]
            self.financials['InterestExpense'] = [{'val': yf_get_latest(income, 'Interest Expense')}]
            self.financials['CashAndEquivalents'] = [{'val': yf_get_latest(bs, 'Cash And Cash Equivalents')}]
            self.financials['Inventory'] = [{'val': yf_get_latest(bs, 'Inventory')}]
            
            # Calculate total debt
            long_debt = yf_get_latest(bs, 'Long Term Debt')
            short_debt = yf_get_latest(bs, 'Current Debt')
            self.financials['DebtCurrent'] = [{'val': short_debt}]
            self.financials['DebtNoncurrent'] = [{'val': long_debt}]
            
            print("    ✓ Successfully loaded financial data from yfinance")
            return True
            
        except Exception as e:
            print(f"    ✗ yfinance error: {e}")
            return False
    
    def fetch_price_data_yfinance(self):
        """Try to fetch prices from yfinance"""
        try:
            import yfinance as yf
            print("  - Trying yfinance...")
            
            end_date = datetime.now()
            start_date = end_date - timedelta(days=5*365)
            
            stock = yf.Ticker(self.ticker)
            df = stock.history(start=start_date, end=end_date, auto_adjust=False)
            
            if df is not None and len(df) > 100:
                self.historical_prices = df
                print(f"    ✓ Got {len(df)} days from yfinance")
                return True
            return False
        except Exception as e:
            print(f"    ✗ yfinance failed: {e}")
            return False
    
    def fetch_price_data_yahoo_direct(self):
        """Direct Yahoo Finance API (no library)"""
        try:
            print("  - Trying Yahoo Finance direct API...")
            
            end = int(datetime.now().timestamp())
            start = int((datetime.now() - timedelta(days=5*365)).timestamp())
            
            url = f"https://query1.finance.yahoo.com/v7/finance/download/{self.ticker}"
            params = {
                'period1': start,
                'period2': end,
                'interval': '1d',
                'events': 'history'
            }
            headers = {'User-Agent': 'Mozilla/5.0'}
            
            time.sleep(1)  # Be nice to Yahoo
            response = requests.get(url, params=params, headers=headers, timeout=10)
            
            if response.status_code == 200:
                df = pd.read_csv(StringIO(response.text))
                df['Date'] = pd.to_datetime(df['Date'])
                df.set_index('Date', inplace=True)
                
                if len(df) > 100:
                    # Rename columns to match yfinance format
                    df.rename(columns={'Adj Close': 'Close'}, inplace=True)
                    self.historical_prices = df
                    print(f"    ✓ Got {len(df)} days from Yahoo direct")
                    return True
            return False
        except Exception as e:
            print(f"    ✗ Yahoo direct failed: {e}")
            return False
    
    def fetch_price_data_alphavantage_free(self):
        """Try Alpha Vantage demo/free endpoint"""
        try:
            print("  - Trying Alpha Vantage free endpoint...")
            
            # Alpha Vantage has a demo key that works for limited requests
            url = "https://www.alphavantage.co/query"
            params = {
                'function': 'TIME_SERIES_DAILY',
                'symbol': self.ticker,
                'outputsize': 'full',
                'apikey': 'demo'  # Limited demo key
            }
            
            time.sleep(2)
            response = requests.get(url, params=params, timeout=15)
            
            if response.status_code == 200:
                data = response.json()
                if 'Time Series (Daily)' in data:
                    prices = data['Time Series (Daily)']
                    df = pd.DataFrame.from_dict(prices, orient='index')
                    df.index = pd.to_datetime(df.index)
                    df = df.sort_index()
                    df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
                    
                    for col in df.columns:
                        df[col] = pd.to_numeric(df[col])
                    
                    cutoff = datetime.now() - timedelta(days=5*365)
                    df = df[df.index >= cutoff]
                    
                    if len(df) > 100:
                        self.historical_prices = df
                        print(f"    ✓ Got {len(df)} days from Alpha Vantage")
                        return True
            return False
        except Exception as e:
            print(f"    ✗ Alpha Vantage failed: {e}")
            return False
    
    def fetch_sec_financials(self):
        """Fetch financial data from SEC Edgar"""
        try:
            if not self.cik:
                if not self.get_cik():
                    return False
            
            print(f"  - Fetching SEC filings for CIK {self.cik}...")
            
            # SEC company facts API (XBRL data)
            url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{self.cik}.json"
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                'Accept': 'application/json',
                'Host': 'data.sec.gov'
            }
            
            time.sleep(0.5)  # Be respectful to SEC servers
            response = requests.get(url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                print(f"    ✗ SEC API returned status {response.status_code}")
                print(f"    URL tried: {url}")
                print(f"    Response: {response.text[:200]}")
                return False
            
            data = response.json()
            
            # Extract financial data
            if 'facts' in data:
                us_gaap = data['facts'].get('us-gaap', {})
                
                # Get the most recent annual filings (10-K)
                def get_recent_values(concept, num_years=5):
                    if concept not in us_gaap:
                        return []
                    
                    units = us_gaap[concept].get('units', {})
                    # Usually in USD
                    usd_data = units.get('USD', [])
                    
                    # Filter for annual reports (10-K)
                    annual = [x for x in usd_data if x.get('form') == '10-K']
                    # Sort by date descending
                    annual.sort(key=lambda x: x.get('end', ''), reverse=True)
                    
                    return annual[:num_years]
                
                # Key financial metrics
                self.financials['Assets'] = get_recent_values('Assets')
                self.financials['AssetsCurrent'] = get_recent_values('AssetsCurrent')
                self.financials['LiabilitiesCurrent'] = get_recent_values('LiabilitiesCurrent')
                self.financials['Liabilities'] = get_recent_values('Liabilities')
                self.financials['StockholdersEquity'] = get_recent_values('StockholdersEquity')
                self.financials['RetainedEarningsAccumulatedDeficit'] = get_recent_values('RetainedEarningsAccumulatedDeficit')
                self.financials['Revenues'] = get_recent_values('Revenues')
                self.financials['OperatingIncomeLoss'] = get_recent_values('OperatingIncomeLoss')
                self.financials['NetIncomeLoss'] = get_recent_values('NetIncomeLoss')
                self.financials['OperatingCashFlow'] = get_recent_values('NetCashProvidedByUsedInOperatingActivities')
                self.financials['DebtCurrent'] = get_recent_values('LongTermDebtCurrent')
                self.financials['DebtNoncurrent'] = get_recent_values('LongTermDebtNoncurrent')
                self.financials['InterestExpense'] = get_recent_values('InterestExpense')
                self.financials['CashAndEquivalents'] = get_recent_values('CashAndCashEquivalentsAtCarryingValue')
                self.financials['Inventory'] = get_recent_values('InventoryNet')
                
                print(f"    ✓ Retrieved SEC financial data")
                return True
            else:
                print("    ✗ No financial facts found in SEC data")
                return False
                
        except Exception as e:
            print(f"    ✗ Error fetching SEC data: {e}")
            return False
    
    def fetch_data(self):
        """Main data fetching with multiple fallbacks"""
        print(f"\nFetching data for {self.ticker}...")
        print("="*60)
        
        # Try SEC first
        print("\n1. Trying SEC Edgar (US companies only):")
        sec_success = self.fetch_sec_financials()
        
        # If SEC fails, try yfinance as complete fallback
        if not sec_success:
            print("\n   Falling back to yfinance for financial statements...")
            if self.fetch_data_yfinance_complete():
                print("   ✓ Using yfinance data instead")
            else:
                print("\n❌ Could not fetch data from any source")
                print("\nTroubleshooting:")
                print("  1. Check if ticker is correct")
                print("  2. For non-US companies, SEC won't work")
                print("  3. Try waiting 30 seconds and run again (rate limits)")
                print("  4. Check your internet connection")
                return False
        
        # Get price data with fallbacks
        print("\n2. Fetching Historical Prices:")
        
        success = (self.fetch_price_data_yfinance() or 
                  self.fetch_price_data_yahoo_direct() or
                  self.fetch_price_data_alphavantage_free())
        
        if not success:
            print("\n⚠️  Could not fetch price data")
            print("   VaR, CVaR, and volatility metrics will be unavailable")
        
        print("\n" + "="*60)
        print("✓ Data collection complete!")
        return True
    
    def get_latest_value(self, key, default=0):
        """Get the most recent value for a financial metric"""
        try:
            if key in self.financials and self.financials[key]:
                return float(self.financials[key][0].get('val', default))
            return default
        except:
            return default
    
    def calculate_altman_z_score(self):
        """Calculate Altman Z-Score"""
        try:
            # Get values from SEC data
            total_assets = self.get_latest_value('Assets')
            current_assets = self.get_latest_value('AssetsCurrent')
            current_liabilities = self.get_latest_value('LiabilitiesCurrent')
            retained_earnings = self.get_latest_value('RetainedEarningsAccumulatedDeficit')
            ebit = self.get_latest_value('OperatingIncomeLoss')
            total_liabilities = self.get_latest_value('Liabilities')
            revenue = self.get_latest_value('Revenues')
            stockholders_equity = self.get_latest_value('StockholdersEquity')
            
            if total_assets == 0:
                print("  ✗ Cannot calculate Altman Z-Score (missing data)")
                return None, "Insufficient data"
            
            # Calculate components
            X1 = (current_assets - current_liabilities) / total_assets
            X2 = retained_earnings / total_assets
            X3 = ebit / total_assets
            
            # For X4, use book value of equity since we may not have market cap
            X4 = stockholders_equity / total_liabilities if total_liabilities > 0 else 0
            X5 = revenue / total_assets
            
            # Altman Z-Score (using book value version)
            z_score = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
            
            # Interpretation
            if z_score > 2.99:
                interpretation = "Safe Zone (Low bankruptcy risk)"
            elif z_score > 1.81:
                interpretation = "Grey Zone (Moderate risk)"
            else:
                interpretation = "Distress Zone (High bankruptcy risk)"
            
            self.results['altman_z_score'] = {
                'score': z_score,
                'interpretation': interpretation,
                'components': {
                    'X1_working_capital_ratio': X1,
                    'X2_retained_earnings_ratio': X2,
                    'X3_ebit_ratio': X3,
                    'X4_equity_ratio': X4,
                    'X5_asset_turnover': X5
                }
            }
            
            print(f"  ✓ Altman Z-Score: {z_score:.2f} ({interpretation})")
            return z_score, interpretation
            
        except Exception as e:
            print(f"  ✗ Error calculating Altman Z-Score: {e}")
            return None, str(e)
    
    def calculate_var_cvar(self, confidence_level=0.95):
        """Calculate VaR and CVaR"""
        try:
            if self.historical_prices is None:
                print("  ⚠️  Skipping VaR/CVaR (no price data)")
                return None, None, "No price data"
            
            returns = self.historical_prices['Close'].pct_change().dropna()
            
            if len(returns) < 50:
                print("  ⚠️  Insufficient return data")
                return None, None, "Insufficient data"
            
            var = np.percentile(returns, (1 - confidence_level) * 100)
            cvar = returns[returns <= var].mean()
            
            var_annual = var * np.sqrt(252)
            cvar_annual = cvar * np.sqrt(252)
            
            self.results['var_cvar'] = {
                'var_daily': var,
                'cvar_daily': cvar,
                'var_annual': var_annual,
                'cvar_annual': cvar_annual,
                'confidence_level': confidence_level
            }
            
            print(f"  ✓ VaR (95%): {abs(var)*100:.2f}% daily, {abs(var_annual)*100:.2f}% annual")
            return var, cvar, "Success"
            
        except Exception as e:
            print(f"  ⚠️  VaR/CVaR calculation skipped: {e}")
            return None, None, str(e)
    
    def calculate_liquidity_ratios(self):
        """Calculate liquidity ratios"""
        try:
            current_assets = self.get_latest_value('AssetsCurrent')
            current_liabilities = self.get_latest_value('LiabilitiesCurrent')
            cash = self.get_latest_value('CashAndEquivalents')
            inventory = self.get_latest_value('Inventory')
            
            current_ratio = current_assets / current_liabilities if current_liabilities > 0 else 0
            quick_ratio = (current_assets - inventory) / current_liabilities if current_liabilities > 0 else 0
            cash_ratio = cash / current_liabilities if current_liabilities > 0 else 0
            
            self.results['liquidity_ratios'] = {
                'current_ratio': current_ratio,
                'quick_ratio': quick_ratio,
                'cash_ratio': cash_ratio
            }
            
            print(f"  ✓ Current Ratio: {current_ratio:.2f}, Quick Ratio: {quick_ratio:.2f}")
            return self.results['liquidity_ratios']
            
        except Exception as e:
            print(f"  ✗ Error calculating liquidity: {e}")
            return None
    
    def calculate_leverage_ratios(self):
        """Calculate leverage ratios"""
        try:
            debt_current = self.get_latest_value('DebtCurrent')
            debt_noncurrent = self.get_latest_value('DebtNoncurrent')
            total_debt = debt_current + debt_noncurrent
            
            total_equity = self.get_latest_value('StockholdersEquity')
            total_assets = self.get_latest_value('Assets')
            ebit = self.get_latest_value('OperatingIncomeLoss')
            interest_expense = self.get_latest_value('InterestExpense')
            
            debt_to_equity = total_debt / total_equity if total_equity > 0 else 0
            debt_to_assets = total_debt / total_assets if total_assets > 0 else 0
            interest_coverage = ebit / abs(interest_expense) if interest_expense != 0 else 0
            
            self.results['leverage_ratios'] = {
                'debt_to_equity': debt_to_equity,
                'debt_to_assets': debt_to_assets,
                'interest_coverage': interest_coverage
            }
            
            print(f"  ✓ Debt/Equity: {debt_to_equity:.2f}, Interest Coverage: {interest_coverage:.2f}x")
            return self.results['leverage_ratios']
            
        except Exception as e:
            print(f"  ✗ Error calculating leverage: {e}")
            return None
    
    def calculate_profitability_ratios(self):
        """Calculate profitability ratios"""
        try:
            revenue = self.get_latest_value('Revenues')
            operating_income = self.get_latest_value('OperatingIncomeLoss')
            net_income = self.get_latest_value('NetIncomeLoss')
            total_assets = self.get_latest_value('Assets')
            total_equity = self.get_latest_value('StockholdersEquity')
            
            operating_margin = operating_income / revenue if revenue > 0 else 0
            net_margin = net_income / revenue if revenue > 0 else 0
            roa = net_income / total_assets if total_assets > 0 else 0
            roe = net_income / total_equity if total_equity > 0 else 0
            
            self.results['profitability_ratios'] = {
                'operating_margin': operating_margin,
                'net_margin': net_margin,
                'roa': roa,
                'roe': roe
            }
            
            print(f"  ✓ Net Margin: {net_margin*100:.2f}%, ROE: {roe*100:.2f}%")
            return self.results['profitability_ratios']
            
        except Exception as e:
            print(f"  ✗ Error calculating profitability: {e}")
            return None
    
    def calculate_credit_score(self):
        """Calculate overall credit score"""
        try:
            score = 50
            
            # Altman Z-Score (0-20 pts)
            if 'altman_z_score' in self.results:
                z = self.results['altman_z_score']['score']
                score += 20 if z > 2.99 else 10 if z > 1.81 else 0
            
            # Liquidity (0-15 pts)
            if 'liquidity_ratios' in self.results:
                cr = self.results['liquidity_ratios']['current_ratio']
                score += 15 if cr > 2 else 10 if cr > 1 else 5
            
            # Leverage (0-20 pts)
            if 'leverage_ratios' in self.results:
                d_to_e = self.results['leverage_ratios']['debt_to_equity']
                int_cov = self.results['leverage_ratios']['interest_coverage']
                score += 10 if d_to_e < 1 else 5 if d_to_e < 2 else 0
                score += 10 if int_cov > 5 else 5 if int_cov > 2.5 else 0
            
            # Profitability (0-15 pts)
            if 'profitability_ratios' in self.results:
                roe = self.results['profitability_ratios']['roe']
                net_margin = self.results['profitability_ratios']['net_margin']
                score += 8 if roe > 0.15 else 5 if roe > 0.10 else 0
                score += 7 if net_margin > 0.15 else 4 if net_margin > 0.05 else 0
            
            score = min(score, 100)
            
            # Rating
            if score >= 85:
                rating = "AAA - Excellent"
            elif score >= 75:
                rating = "AA - Very Good"
            elif score >= 65:
                rating = "A - Good"
            elif score >= 55:
                rating = "BBB - Adequate"
            elif score >= 45:
                rating = "BB - Moderate Risk"
            else:
                rating = "B - High Risk"
            
            self.results['credit_score'] = {
                'score': score,
                'rating': rating
            }
            
            print(f"  ✓ Credit Score: {score:.0f}/100 ({rating})")
            return score, rating
            
        except Exception as e:
            print(f"  ✗ Error calculating credit score: {e}")
            return None, str(e)
    
    def run_full_analysis(self):
        """Run complete analysis"""
        if not self.fetch_data():
            return None
        
        print("\n" + "="*60)
        print(f"CALCULATING CREDIT METRICS: {self.ticker}")
        print("="*60 + "\n")
        
        print("1. Altman Z-Score")
        self.calculate_altman_z_score()
        
        print("\n2. VaR and CVaR (95% confidence)")
        self.calculate_var_cvar()
        
        print("\n3. Liquidity Ratios")
        self.calculate_liquidity_ratios()
        
        print("\n4. Leverage Ratios")
        self.calculate_leverage_ratios()
        
        print("\n5. Profitability Ratios")
        self.calculate_profitability_ratios()
        
        print("\n6. Overall Credit Score")
        self.calculate_credit_score()
        
        print("\n" + "="*60)
        print("✓ ANALYSIS COMPLETE")
        print("="*60)
        
        return self.results
    
    def print_summary(self):
        """Print detailed summary"""
        if not self.results:
            print("No results available")
            return
        
        print("\n" + "="*60)
        print(f"CREDIT ANALYSIS SUMMARY: {self.ticker}")
        print("="*60)
        
        if 'credit_score' in self.results:
            print(f"\n📊 OVERALL CREDIT SCORE: {self.results['credit_score']['score']:.0f}/100")
            print(f"   Rating: {self.results['credit_score']['rating']}")
        
        if 'altman_z_score' in self.results:
            z = self.results['altman_z_score']
            print(f"\n📈 ALTMAN Z-SCORE: {z['score']:.2f}")
            print(f"   {z['interpretation']}")
        
        if 'var_cvar' in self.results:
            var = self.results['var_cvar']
            print(f"\n⚠️  VALUE AT RISK (95% confidence):")
            print(f"   Daily VaR: {abs(var['var_daily'])*100:.2f}%")
            print(f"   Annual VaR: {abs(var['var_annual'])*100:.2f}%")
            print(f"   Daily CVaR: {abs(var['cvar_daily'])*100:.2f}%")
        
        if 'liquidity_ratios' in self.results:
            liq = self.results['liquidity_ratios']
            print(f"\n💧 LIQUIDITY:")
            print(f"   Current Ratio: {liq['current_ratio']:.2f}")
            print(f"   Quick Ratio: {liq['quick_ratio']:.2f}")
        
        if 'leverage_ratios' in self.results:
            lev = self.results['leverage_ratios']
            print(f"\n⚖️  LEVERAGE:")
            print(f"   Debt/Equity: {lev['debt_to_equity']:.2f}")
            print(f"   Interest Coverage: {lev['interest_coverage']:.2f}x")
        
        if 'profitability_ratios' in self.results:
            prof = self.results['profitability_ratios']
            print(f"\n💰 PROFITABILITY:")
            print(f"   Net Margin: {prof['net_margin']*100:.2f}%")
            print(f"   ROE: {prof['roe']*100:.2f}%")
        
        print("\n" + "="*60)


# ==============================================================================
# EXAMPLE USAGE - 100% FREE!
# ==============================================================================

def test_sec_connection(ticker):
    """Test SEC connection and show what's happening"""
    print(f"\n🔍 Testing SEC connection for {ticker}...")
    print("="*60)
    
    # Step 1: Get CIK
    print("\nStep 1: Getting CIK...")
    try:
        url = "https://www.sec.gov/files/company_tickers.json"
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        print(f"  Status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            cik = None
            for item in data.values():
                if item['ticker'].upper() == ticker.upper():
                    cik = str(item['cik_str']).zfill(10)
                    print(f"  ✓ Found CIK: {cik}")
                    break
            
            if not cik:
                print(f"  ✗ Ticker {ticker} not found")
                return
            
            # Step 2: Test SEC API
            print("\nStep 2: Testing SEC API...")
            url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                'Accept': 'application/json',
                'Host': 'data.sec.gov'
            }
            
            time.sleep(0.5)
            response = requests.get(url, headers=headers, timeout=15)
            print(f"  Status: {response.status_code}")
            
            if response.status_code == 200:
                data = response.json()
                if 'facts' in data and 'us-gaap' in data['facts']:
                    concepts = list(data['facts']['us-gaap'].keys())
                    print(f"  ✓ SEC data available! Found {len(concepts)} financial concepts")
                    print(f"  Sample concepts: {concepts[:5]}")
                    print("\n✅ Connection successful! You can proceed with analysis.")
                else:
                    print("  ✗ No GAAP data found")
            else:
                print(f"  ✗ Failed to fetch data")
                print(f"  Response: {response.text[:500]}")
                
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    print("="*60)


if __name__ == "__main__":
    print("="*60)
    print("CREDIT WORTHINESS MODEL - 100% FREE VERSION")
    print("="*60)
    print("\nNo API key needed!")
    print("Uses: SEC Edgar (financials) + Multiple free price sources")
    print("\nNote: Only works for US companies that file with SEC")
    print("="*60)
    
    # First, test the connection
    ticker = "AAPL"  # Try: MSFT, GOOGL, JPM, TSLA, etc.
    test_sec_connection(ticker)
    
    # Then run the full analysis
    print("\n\n" + "="*60)
    print("RUNNING FULL ANALYSIS")
    print("="*60)
    
    model = CreditWorthinessModel(ticker)
    results = model.run_full_analysis()
    
    if results:
        model.print_summary()
        
        # Export to DataFrame
        data = []
        for category, metrics in results.items():
            if isinstance(metrics, dict) and category not in ['credit_score']:
                for key, value in metrics.items():
                    if key not in ['interpretation', 'components'] and not isinstance(value, dict):
                        data.append({
                            'Category': category.replace('_', ' ').title(),
                            'Metric': key.replace('_', ' ').title(),
                            'Value': f"{value:.4f}" if isinstance(value, float) else value
                        })
        
        if data:
            df = pd.DataFrame(data)
            print("\n" + "="*60)
            print("DETAILED METRICS")
            print("="*60)
            print(df.to_string(index=False))

CREDIT WORTHINESS MODEL - 100% FREE VERSION

No API key needed!
Uses: SEC Edgar (financials) + Multiple free price sources

Note: Only works for US companies that file with SEC

🔍 Testing SEC connection for AAPL...

Step 1: Getting CIK...
  Status: 403


RUNNING FULL ANALYSIS

Fetching data for AAPL...

1. Trying SEC Edgar (US companies only):

   Falling back to yfinance for financial statements...
  - Attempting yfinance for financial statements...
    ✗ yfinance: No financial data

❌ Could not fetch data from any source

Troubleshooting:
  1. Check if ticker is correct
  2. For non-US companies, SEC won't work
  3. Try waiting 30 seconds and run again (rate limits)
  4. Check your internet connection


In [19]:
import pandas as pd
import numpy as np

# Replace 'BHP' with your stock ticker / filename prefix
ticker = "BHP"

bs = pd.read_csv(f"{ticker}_BalanceSheet.csv", index_col=0)
is_ = pd.read_csv(f"{ticker}_IncomeStatement.csv", index_col=0)
cf = pd.read_csv(f"{ticker}_Cashflow.csv", index_col=0)
prices = pd.read_csv(f"{ticker}_Prices.csv", index_col=0, parse_dates=True)

print("Balance Sheet sample:\n", bs.head())
print("Income Statement sample:\n", is_.head())
print("Cash Flow sample:\n", cf.head())
print("Price data sample:\n", prices.head())


# Altmann Z score calculations
def calc_altman_z(bs, is_):
    try:
        ta = bs.loc["Total Assets"].iloc[0]
        tl = bs.loc["Total Liab"].iloc[0]
        wc = bs.loc["Total Current Assets"].iloc[0] - bs.loc["Total Current Liabilities"].iloc[0]
        re = bs.loc["Retained Earnings"].iloc[0]
        ebit = is_.loc["Ebit"].iloc[0]
        s = is_.loc["Total Revenue"].iloc[0]

        z = 1.2*(wc/ta) + 1.4*(re/ta) + 3.3*(ebit/ta) + 0.999*(s/ta)
        return z
    except:
        return np.nan

    
# Piotroski F score calculations
def calc_piotroski_f(bs, is_, cf):
    try:
        score = 0
        ni = is_.loc["Net Income"].iloc[0]
        ni_prev = is_.loc["Net Income"].iloc[1]
        roa = ni / bs.loc["Total Assets"].iloc[0]
        roa_prev = ni_prev / bs.loc["Total Assets"].iloc[1]
        cfo = cf.loc["Total Cash From Operating Activities"].iloc[0]

        score += (ni > 0)
        score += (roa > roa_prev)
        score += (cfo > 0)
        score += (cfo > ni)

        ltd = bs.loc["Long Term Debt"].iloc[0]
        ltd_prev = bs.loc["Long Term Debt"].iloc[1]
        score += (ltd <= ltd_prev)

        cr = bs.loc["Total Current Assets"].iloc[0] / bs.loc["Total Current Liabilities"].iloc[0]
        cr_prev = bs.loc["Total Current Assets"].iloc[1] / bs.loc["Total Current Liabilities"].iloc[1]
        score += (cr > cr_prev)

        gm = is_.loc["Gross Profit"].iloc[0] / is_.loc["Total Revenue"].iloc[0]
        gm_prev = is_.loc["Gross Profit"].iloc[1] / is_.loc["Total Revenue"].iloc[1]
        score += (gm > gm_prev)

        at = is_.loc["Total Revenue"].iloc[0] / bs.loc["Total Assets"].iloc[0]
        at_prev = is_.loc["Total Revenue"].iloc[1] / bs.loc["Total Assets"].iloc[1]
        score += (at > at_prev)

        return score
    except:
        return np.nan

    
# Debt ratios and other metrics
def compute_credit_metrics(bs, is_, cf):
    metrics = {}
    debt = bs.loc["Total Debt"].iloc[0]
    assets = bs.loc["Total Assets"].iloc[0]
    ebit = is_.loc["Ebit"].iloc[0]
    interest = abs(is_.loc["Interest Expense"].iloc[0])
    cfo = cf.loc["Total Cash From Operating Activities"].iloc[0]

    metrics["Debt/Assets"] = debt / assets
    metrics["Interest Coverage"] = ebit / interest if interest != 0 else np.nan
    metrics["CFO/Debt"] = cfo / debt if debt != 0 else np.nan
    metrics["Current Ratio"] = bs.loc["Total Current Assets"].iloc[0] / bs.loc["Total Current Liabilities"].iloc[0]
    return metrics


# VaR calculation !!!!!!Make sure your prices CSV column is named Adj Close
def get_var_from_prices(prices, confidence=0.95):
    returns = prices['Adj Close'].pct_change().dropna()
    return np.percentile(returns, (1-confidence)*100)


# Combination of Score & Rating
def score_model(metrics, z, f, var):
    score = 0
    score += (1 - metrics["Debt/Assets"]) * 20
    score += min(metrics["Interest Coverage"]/10, 1) * 25
    score += min(metrics["CFO/Debt"], 1) * 25
    score += min(metrics["Current Ratio"]/2, 1) * 15
    score += (z/5) * 10
    score += (f/9) * 5
    score -= abs(var) * 10
    return round(max(min(score*100, 100), 0), 2)

def rating(score):
    if score >= 90: return "AAA"
    if score >= 80: return "AA"
    if score >= 70: return "A"
    if score >= 60: return "BBB"
    if score >= 50: return "BB"
    if score >= 40: return "B"
    return "CCC or lower"


# Run the model
z = calc_altman_z(bs, is_)
f = calc_piotroski_f(bs, is_, cf)
var = get_var_from_prices(prices)
metrics = compute_credit_metrics(bs, is_, cf)

score = score_model(metrics, z, f, var)
rating_grade = rating(score)

# Print results
print(f"\nCredit Score for {ticker}: {score}/100 — Rating: {rating_grade}\n")
print("Metrics:")
for k,v in metrics.items():
    print(f"{k}: {v:.3f}")
print(f"\nAltman Z-Score: {z:.2f}")
print(f"Piotroski F-Score: {f}")
print(f"VaR: {var:.2%}")

FileNotFoundError: [Errno 2] No such file or directory: 'BHP_BalanceSheet.csv'